# Sentinel — QLoRA fine-tune of the alert triage classifier

Trains `Qwen2.5-1.5B-Instruct` to classify production alerts into a fixed
JSON schema, then exports it for Ollama.

**Runtime → Change runtime type → T4 GPU.** Roughly 25–35 minutes on a free T4.

## Why this notebook exists

This project was built on a machine with a 6GB RTX 3060, and the trainer in
`finetune/train_qlora.py` targets exactly that. It cannot run there: Windows
Smart App Control blocks the CUDA build of PyTorch (`shm.dll` and its
dependencies are unsigned), and disabling Smart App Control on Windows 11 is
irreversible without reinstalling the OS. The CPU wheel loads fine but would
turn a 30-minute run into an overnight one.

So the fine-tune runs here. The hyperparameters are identical to the local
script — a T4 has 16GB against the 3060's 6GB, so anything that fits locally
fits here with room to spare.

## 1. Environment

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
%pip install -q -U "transformers>=4.48" "peft>=0.14" "trl>=0.13" \
    "datasets>=3.2" "accelerate>=1.2" "bitsandbytes>=0.45" sentencepiece

## 2. Get the dataset

Generated from templates rather than distilled from a frontier model. These
labels are definitional — a total outage is P1 because of what the alert says —
so template generation makes every label correct by construction and the whole
set reproducible from a seed.

Clone the repo to regenerate it, or upload `train_chat.jsonl` / `val_chat.jsonl`
from `finetune/data/` directly.

In [ ]:
import os

REPO = "https://github.com/GautamRaju18/Sentinel.git"

if not os.path.exists("Sentinel"):
    !git clone -q $REPO
%cd Sentinel
%pip install -q pydantic pydantic-settings
!python finetune/generate_dataset.py

In [ ]:
import json

def load(name):
    with open(f"finetune/data/{name}_chat.jsonl", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

train_rows, val_rows = load("train"), load("val")
print(f"train={len(train_rows)}  val={len(val_rows)}")
print(json.dumps(train_rows[0], indent=2)[:900])

## 3. Load the base model in 4-bit

NF4 with double quantisation. The base weights are frozen, so their precision
matters far less than the adapter's; double-quant saves roughly 0.4 bits per
parameter, which is what makes this fit in 6GB on the target hardware.

In [ ]:
import torch
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

BASE = "Qwen/Qwen2.5-1.5B-Instruct"

# Pick precision from what the GPU actually supports.
#   T4 (compute capability 7.5) has NO bf16 support -> fp16 + GradScaler
#   A100 / L4 / newer (8.0+)     -> bf16, no scaler needed
# Qwen2.5's config declares bfloat16, so on a T4 the model would load as bf16
# and then fp16 training raises:
#   "_amp_foreach_non_finite_check_and_unscale_cuda" not implemented for 'BFloat16'
USE_BF16 = torch.cuda.is_bf16_supported()
DTYPE = torch.bfloat16 if USE_BF16 else torch.float16
print(f"GPU bf16 support: {USE_BF16}  ->  training dtype {DTYPE}")

# transformers 5.x renamed `torch_dtype` to `dtype`. from_pretrained swallows
# unknown kwargs, so passing the wrong name fails SILENTLY and the config
# default wins — which is exactly how the bf16 mismatch above happened.
DTYPE_KW = "dtype" if int(transformers.__version__.split(".")[0]) >= 5 else "torch_dtype"
print(f"transformers {transformers.__version__} -> using `{DTYPE_KW}=`")

tokenizer = AutoTokenizer.from_pretrained(BASE, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

quant = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=DTYPE,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    BASE,
    quantization_config=quant,
    device_map={"": 0},
    trust_remote_code=True,
    **{DTYPE_KW: DTYPE},
)
model.config.use_cache = False

# Verify the dtype actually applied rather than trusting that it did.
actual = next(p.dtype for p in model.parameters() if p.is_floating_point())
print(f"loaded. param dtype: {actual}  |  GPU memory: {torch.cuda.memory_allocated()/1e9:.2f} GB")
assert actual == DTYPE, f"dtype did not apply: wanted {DTYPE}, got {actual}"

## 4. Baseline: what does the model do before training?

Worth running. Without it you have no idea whether the fine-tune helped or
whether the base model could already do this.

In [ ]:
def classify(messages, max_new_tokens=140):
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=max_new_tokens,
            do_sample=False, pad_token_id=tokenizer.pad_token_id,
        )
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

sample = train_rows[0]["messages"]
print("--- before training ---")
print(classify(sample[:2]))
print("\n--- ground truth ---")
print(sample[2]["content"])

## 5. Attach the LoRA adapter

Rank 16 on attention **and** MLP projections. Attention-only adapters learn the
task but keep drifting on output format; the MLP projections are where the
"always emit this JSON shape" behaviour settles.

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

peft_config = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
)
model = get_peft_model(model, peft_config)

# Keep trainable params in fp32. Under fp16 training the GradScaler unscales
# gradients, and it only implements that for fp32/fp16 — a half-precision
# adapter parameter makes it raise. fp32 adapters are also just better
# numerically, and at rank 16 they cost a few MB.
if not USE_BF16:
    for param in model.parameters():
        if param.requires_grad:
            param.data = param.data.to(torch.float32)

model.print_trainable_parameters()
dtypes = {p.dtype for p in model.parameters() if p.requires_grad}
print("trainable param dtypes:", dtypes)

## 6. Train

Loss on the **completion only**. Training on the prompt tokens too would spend
most of the gradient budget teaching the model to reproduce alert text it will
always be given.

In [ ]:
import inspect

from datasets import Dataset
from trl import SFTConfig, SFTTrainer

# TRL's SFT API has moved more than once: `max_seq_length` became `max_length`,
# and `DataCollatorForCompletionOnlyLM` was removed in favour of the
# `assistant_only_loss` flag. Colab installs whatever is current, so ask this
# installation what it accepts rather than pinning a version that will rot.
sft_params = set(inspect.signature(SFTConfig.__init__).parameters)
modern = "assistant_only_loss" in sft_params
print("TRL API:", "modern (assistant_only_loss)" if modern else "legacy (collator)")

kwargs = {
    "output_dir": "outputs/triage-qwen1.5b-lora",
    "num_train_epochs": 3,
    "per_device_train_batch_size": 2,
    "gradient_accumulation_steps": 4,
    "gradient_checkpointing": True,
    "learning_rate": 2e-4,
    "lr_scheduler_type": "cosine",
    "warmup_ratio": 0.03,
    "logging_steps": 10,
    "eval_strategy": "steps",
    "eval_steps": 50,
    "save_strategy": "steps",
    "save_steps": 100,
    "save_total_limit": 2,
    # Must match the dtype the model was actually loaded in — see cell 3.
    "bf16": USE_BF16,
    "fp16": not USE_BF16,
    "optim": "paged_adamw_8bit",
    # packing would blur example boundaries, which matters for classification
    "packing": False,
    "report_to": [],
}
kwargs["max_length" if "max_length" in sft_params else "max_seq_length"] = 1024

if modern:
    # Newer TRL applies the chat template itself and masks every non-assistant
    # token — the same effect the old collator gave, so pass raw messages.
    kwargs["assistant_only_loss"] = True
    train_ds = Dataset.from_list([{"messages": r["messages"]} for r in train_rows])
    eval_ds = Dataset.from_list([{"messages": r["messages"]} for r in val_rows])
else:
    kwargs["dataset_text_field"] = "text"

    def to_dataset(rows):
        return Dataset.from_dict({
            "text": [tokenizer.apply_chat_template(r["messages"], tokenize=False) for r in rows]
        })

    train_ds, eval_ds = to_dataset(train_rows), to_dataset(val_rows)

config = SFTConfig(**{k: v for k, v in kwargs.items() if k in sft_params})
print(f"precision: bf16={config.bf16} fp16={config.fp16}")

trainer = SFTTrainer(
    model=model, args=config,
    train_dataset=train_ds, eval_dataset=eval_ds,
    processing_class=tokenizer,
)

if not modern:
    from trl import DataCollatorForCompletionOnlyLM
    trainer.data_collator = DataCollatorForCompletionOnlyLM(
        response_template="<|im_start|>assistant\n", tokenizer=tokenizer
    )
    print("completion-only loss: collator")

trainer.train()

## 7. After training

In [ ]:
print("--- after training ---")
print(classify(sample[:2]))
print("\n--- ground truth ---")
print(sample[2]["content"])

## 8. Score the held-out test set

The test split uses services that appear in **no** training example, so a model
that memorised service names scores badly here. That is the point of it.

`critical_underestimates` — a P1 filed as P3 or P4 — is the number that matters
operationally. An unattended P1 is an outage nobody is working on; the reverse
merely wastes attention.

In [ ]:
import json, re, time

with open("finetune/data/test.jsonl", encoding="utf-8") as f:
    test_rows = [json.loads(line) for line in f if line.strip()]

SYSTEM = train_rows[0]["messages"][0]["content"]
RANK = {"P1": 0, "P2": 1, "P3": 2, "P4": 3}

def extract_json(text):
    m = re.search(r"\{.*\}", text, re.DOTALL)
    if not m:
        return None
    try:
        return json.loads(m.group(0))
    except json.JSONDecodeError:
        return None

sev_ok = cat_ok = parsed = critical = 0
latencies = []

for row in test_rows:
    started = time.perf_counter()
    raw = classify([
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": f"Classify this alert:\n\n{row['alert']}"},
    ])
    latencies.append((time.perf_counter() - started) * 1000)
    pred, truth = extract_json(raw), row["label"]
    if not pred:
        continue
    parsed += 1
    if pred.get("severity") == truth["severity"]:
        sev_ok += 1
    if pred.get("category") == truth["category"]:
        cat_ok += 1
    if pred.get("severity") in RANK and RANK[pred["severity"]] - RANK[truth["severity"]] >= 2:
        critical += 1

n = len(test_rows)
latencies.sort()
print(f"n                        {n}")
print(f"valid JSON               {parsed/n*100:.1f}%")
print(f"severity accuracy        {sev_ok/n*100:.1f}%   (baseline 37.5%)")
print(f"category accuracy        {cat_ok/n*100:.1f}%   (baseline 25.0%)")
print(f"critical underestimates  {critical/n*100:.1f}%   (baseline 10.0%)")
print(f"latency p50              {latencies[len(latencies)//2]:.0f} ms")

## 9. Merge and export

Merge into **fp16**, not back into 4-bit — merging into quantised weights loses
most of what was learned.

In [ ]:
import gc
import glob
import json
import os
import subprocess

import torch
import transformers
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

BASE = "Qwen/Qwen2.5-1.5B-Instruct"
MERGED = "/content/merged/triage-qwen1.5b"

# PEFT raises on an old torchao rather than treating it as absent. Colab ships
# 0.10.0 and PEFT wants >0.16.0; nothing here uses torchao. --no-deps so a
# different torch is not pulled out from under a live session.
%pip install -q -U --no-deps torchao

# Locate the adapter rather than assuming a relative path. A kernel restart
# resets the working directory to /content, so `outputs/...` silently points
# somewhere that does not exist — which reads as "training was lost" when the
# files are sitting right there.
hits = sorted(glob.glob("/content/**/adapter_config.json", recursive=True))
hits = [h for h in hits if "/drive/" not in h]
if not hits:
    raise SystemExit(
        "No adapter found under /content. The runtime was recycled — re-run the "
        "training cells above."
    )
ADAPTER = os.path.dirname(hits[0])
print("adapter:", ADAPTER)

# Back up BEFORE merging. The adapter is ~35MB and took ~30 minutes to produce;
# leaving it only on ephemeral disk while doing several more minutes of work is
# how a runtime timeout costs a whole training run.
if os.path.isdir("/content/drive/MyDrive"):
    dst = "/content/drive/MyDrive/sentinel-triage-lora"
    os.makedirs(dst, exist_ok=True)
    subprocess.run(f"cp -r {ADAPTER}/* {dst}/", shell=True, check=True)
    print("backed up to", dst)
else:
    print("Drive not mounted — skipping backup. Mount it if you value the adapter:")
    print("  from google.colab import drive; drive.mount('/content/drive')")

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

tokenizer = AutoTokenizer.from_pretrained(BASE, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Merge into fp16, not back into 4-bit — merging into quantised weights loses
# most of what was learned. Same kwarg rename as cell 3: passing torch_dtype to
# a transformers 5.x build is silently ignored.
DTYPE_KW = "dtype" if int(transformers.__version__.split(".")[0]) >= 5 else "torch_dtype"

base = AutoModelForCausalLM.from_pretrained(
    BASE, device_map="cpu", trust_remote_code=True, **{DTYPE_KW: torch.float16}
)
merged = PeftModel.from_pretrained(base, ADAPTER).merge_and_unload()
os.makedirs(MERGED, exist_ok=True)
merged.save_pretrained(MERGED, safe_serialization=True)
tokenizer.save_pretrained(MERGED)
print("merged ->", next(merged.parameters()).dtype)

# transformers 5.x WRITES extra_special_tokens as a list and then REJECTS it as
# one on load ("'list' object has no attribute 'keys'"). A round-trip bug in
# transformers itself. Left unpatched, the GGUF converter dies loading the
# tokenizer, several minutes after the expensive part already succeeded.
cfg_path = f"{MERGED}/tokenizer_config.json"
cfg = json.load(open(cfg_path, encoding="utf-8"))
if not isinstance(cfg.get("extra_special_tokens"), dict):
    cfg["extra_special_tokens"] = {}
    json.dump(cfg, open(cfg_path, "w", encoding="utf-8"), indent=2, ensure_ascii=False)
    print("patched extra_special_tokens -> {}")

# Prove the tokenizer reloads before handing it to the converter.
check = AutoTokenizer.from_pretrained(MERGED, trust_remote_code=True)
print("tokenizer reloads OK, vocab:", len(check))

In [ ]:
import os
import subprocess
import sys

GGUF = "/content/sentinel-triage-f16.gguf"

# Absolute paths throughout. Relative ones break the moment the kernel restarts
# and the working directory reverts to /content.
if not os.path.isdir("/content/llama.cpp"):
    subprocess.run(
        "git clone -q --depth 1 https://github.com/ggerganov/llama.cpp /content/llama.cpp",
        shell=True, check=True,
    )

# Install only what the converter needs. llama.cpp's full requirements file
# tries to pin numpy and protobuf, which conflicts loudly with the versions
# Colab's preinstalled stack depends on.
subprocess.run(f"{sys.executable} -m pip install -q gguf sentencepiece protobuf", shell=True)

subprocess.run(
    f"{sys.executable} /content/llama.cpp/convert_hf_to_gguf.py "
    f"/content/merged/triage-qwen1.5b --outfile {GGUF} --outtype f16",
    shell=True, check=True,
)
print(f"{GGUF}  {os.path.getsize(GGUF) / 1e9:.2f} GB")

# Keep a copy on Drive — the download below can fail on a flaky connection and
# re-running everything to regenerate a 3GB file is a poor use of an afternoon.
if os.path.isdir("/content/drive/MyDrive"):
    subprocess.run(f"cp {GGUF} /content/drive/MyDrive/", shell=True, check=True)
    print("copied to Drive")

## 10. Download and register locally

Download the GGUF, then on your own machine:

```bash
ollama create sentinel-triage -f Modelfile
```

with a `Modelfile` containing:

```
FROM ./sentinel-triage-f16.gguf
PARAMETER temperature 0
PARAMETER top_p 0.1
PARAMETER num_ctx 2048
PARAMETER stop "<|im_end|>"
```

Then point the triage tier at it in `.env`:

```
MODEL_TRIAGE=ollama:sentinel-triage
TRIAGE_BACKEND=finetuned
```

and produce the comparison against the recorded baseline:

```bash
uv run python evals/triage_eval.py --out evals/results_finetuned.json
uv run python evals/compare.py
```

In [ ]:
from google.colab import files

files.download("sentinel-triage-f16.gguf")